# Lookup Table Training — Learned Joint Relationship Model

Trains a shared Transformer encoder with NT-Xent contrastive loss to produce a
`[24, 24, 64]` learned lookup table replacing the 6 hand-crafted features.

In [ ]:
import sys
sys.path.insert(0, '..')   # make project-level imports available

import os
import math
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from scipy.signal import correlate
from tqdm.auto import tqdm

# ── paths ────────────────────────────────────────────────────────────────────
TRAIN_DIR  = Path('../parsed_30gigs/DIP_IMU_train_real_imu_position_only')
OUTPUT_DIR = Path('.')   # save lookup tables next to this notebook

train_files = sorted(TRAIN_DIR.glob('*.pt'))
print(f'Training files found: {len(train_files)}')

# ── SMPL joint names ─────────────────────────────────────────────────────────
JOINT_NAMES = [
    'pelvis',        # 0
    'left_hip',      # 1
    'right_hip',     # 2
    'spine1',        # 3
    'left_knee',     # 4
    'right_knee',    # 5
    'spine2',        # 6
    'left_ankle',    # 7
    'right_ankle',   # 8
    'spine3',        # 9
    'left_foot',     # 10
    'right_foot',    # 11
    'neck',          # 12
    'left_collar',   # 13
    'right_collar',  # 14
    'head',          # 15
    'left_shoulder', # 16
    'right_shoulder',# 17
    'left_elbow',    # 18
    'right_elbow',   # 19
    'left_wrist',    # 20
    'right_wrist',   # 21
    'left_hand',     # 22
    'right_hand',    # 23
]
N_JOINTS = len(JOINT_NAMES)   # 24
N_FEATURES = 6
T = 300

---
## Part 3 — Learned Joint Relationship Model

Replace the 6 hand-crafted features with a learned 64-D embedding using a shared 4-head Transformer encoder and a contrastive (NT-Xent) training objective.

**Architecture:**
- `JointEncoder`: projects `[T, 9]` → `[64]` via a 4-head Transformer + mean pooling
- `PairRelationshipHead`: combines two joint embeddings → `[64]` relationship vector
- Positive pairs for contrastive loss: kinematically adjacent joints from the SMPL skeleton graph

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ── SMPL skeleton adjacency (used to define positive pairs for contrastive loss) ──
SMPL_EDGES = [
    (0, 1), (0, 2), (0, 3),
    (1, 4), (2, 5), (3, 6),
    (4, 7), (5, 8), (6, 9),
    (7, 10), (8, 11), (9, 12),
    (9, 13), (9, 14),
    (12, 15),
    (13, 16), (14, 17),
    (16, 18), (17, 19),
    (18, 20), (19, 21),
    (20, 22), (21, 23),
]

# symmetric adjacency set: (i,j) and (j,i) both included
ADJACENT = {(i, j) for i, j in SMPL_EDGES} | {(j, i) for i, j in SMPL_EDGES}


# ── Sinusoidal positional encoding ───────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 300, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # [1, max_len, d_model]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, d_model]
        return self.dropout(x + self.pe[:, :x.size(1)])


# ── Shared joint encoder ──────────────────────────────────────────────────────
class JointEncoder(nn.Module):
    """
    Maps a single joint's time series [B, T, 9] to a fixed embedding [B, d_embed].
    Weights are shared across all joints — joint identity is not an input.
    """
    def __init__(self, in_channels: int = 9, d_model: int = 128,
                 n_heads: int = 4, n_layers: int = 2,
                 d_embed: int = 64, dropout: float = 0.1):
        super().__init__()
        self.input_proj = nn.Linear(in_channels, d_model)
        self.pos_enc    = PositionalEncoding(d_model, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.pool_proj   = nn.Linear(d_model, d_embed)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, 9]
        x = self.pos_enc(self.input_proj(x))   # [B, T, d_model]
        x = self.transformer(x)                # [B, T, d_model]
        x = x.mean(dim=1)                      # [B, d_model]  — mean pooling over time
        return F.normalize(self.pool_proj(x), dim=-1)   # [B, d_embed], L2-normalised


# ── Pairwise relationship head ────────────────────────────────────────────────
class PairRelationshipHead(nn.Module):
    """
    Combines two L2-normalised joint embeddings into a relationship vector.
    Uses concat + difference + elementwise-product so the head sees both
    absolute position in embedding space and relative geometry.
    """
    def __init__(self, d_embed: int = 64, d_out: int = 64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(d_embed * 4, 256),
            nn.ReLU(),
            nn.Linear(256, d_out),
        )

    def forward(self, e_i: torch.Tensor, e_j: torch.Tensor) -> torch.Tensor:
        # e_i, e_j: [B, d_embed]
        combined = torch.cat([e_i, e_j, (e_i - e_j).abs(), e_i * e_j], dim=-1)
        return F.normalize(self.mlp(combined), dim=-1)   # [B, d_out]


# ── Full model ────────────────────────────────────────────────────────────────
class JointRelationshipModel(nn.Module):
    def __init__(self, d_embed: int = 64):
        super().__init__()
        self.encoder = JointEncoder(d_embed=d_embed)
        self.rel_head = PairRelationshipHead(d_embed=d_embed, d_out=d_embed)

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode a single joint time series. x: [B, T, 9] → [B, d_embed]"""
        return self.encoder(x)

    def forward(self, x_i: torch.Tensor, x_j: torch.Tensor) -> torch.Tensor:
        """Encode a joint pair. x_i, x_j: [B, T, 9] → relationship [B, d_embed]"""
        return self.rel_head(self.encoder(x_i), self.encoder(x_j))

    @torch.no_grad()
    def build_lookup_table(self, vimu: torch.Tensor) -> torch.Tensor:
        """
        Build a [24, 24, d_embed] lookup table from a single sequence.
        vimu: [T, 24, 9]
        """
        T, N, C = vimu.shape
        embeddings = self.encoder(vimu.permute(1, 0, 2))   # [24, T, 9] → [24, d_embed]
        table = torch.zeros(N, N, embeddings.size(-1))
        for i in range(N):
            for j in range(N):
                table[i, j] = self.rel_head(
                    embeddings[i].unsqueeze(0),
                    embeddings[j].unsqueeze(0),
                ).squeeze(0)
        return table


# ── NT-Xent contrastive loss ──────────────────────────────────────────────────
class NTXentLoss(nn.Module):
    """
    Contrastive loss over a batch of joint embedding pairs.
    Positive pairs: kinematically adjacent joints (SMPL_EDGES).
    Negatives: all other pairs in the batch.
    """
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings: torch.Tensor, joint_indices: torch.Tensor) -> torch.Tensor:
        """
        embeddings:    [B * 24, d_embed]  — one embedding per joint per sequence in batch
        joint_indices: [B * 24]           — which joint (0-23) each embedding belongs to
        """
        n = embeddings.size(0)
        sim = torch.mm(embeddings, embeddings.T) / self.temperature   # [n, n]

        # positive mask: same sequence AND kinematically adjacent joints
        pos_mask = torch.zeros(n, n, dtype=torch.bool, device=embeddings.device)
        for idx_a in range(n):
            for idx_b in range(n):
                ji, jj = int(joint_indices[idx_a]), int(joint_indices[idx_b])
                if idx_a != idx_b and (ji, jj) in ADJACENT:
                    pos_mask[idx_a, idx_b] = True

        # mask out self-similarity on diagonal
        eye = torch.eye(n, dtype=torch.bool, device=embeddings.device)
        sim = sim.masked_fill(eye, float('-inf'))

        # for each anchor, pull positives, push negatives
        loss = 0.0
        count = 0
        for i in range(n):
            pos_idx = pos_mask[i].nonzero(as_tuple=True)[0]
            if len(pos_idx) == 0:
                continue
            log_denom = torch.logsumexp(sim[i], dim=0)
            loss += (log_denom - sim[i, pos_idx]).mean()
            count += 1

        return loss / max(count, 1)


# ── Instantiate ───────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model   = JointRelationshipModel(d_embed=64).to(device)
loss_fn = NTXentLoss(temperature=0.07)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Device  : {device}')
print(f'Params  : {n_params:,}')
print(f'Encoder : {sum(p.numel() for p in model.encoder.parameters()):,}')
print(f'RelHead : {sum(p.numel() for p in model.rel_head.parameters()):,}')
print()
print(model)

In [ ]:
import random
import time
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
N_EPOCHS    = 30
BATCH_SIZE  = 8
LR          = 3e-4
VAL_SPLIT   = 0.1        # 10% of files held out for validation
CKPT_PATH   = OUTPUT_DIR / 'joint_rel_model_best.pt'
TEMPERATURE = 0.07

# ── Train / val split ─────────────────────────────────────────────────────────
random.seed(42)
all_files  = list(train_files)
random.shuffle(all_files)
n_val      = max(1, int(len(all_files) * VAL_SPLIT))
val_files  = all_files[:n_val]
trn_files  = all_files[n_val:]
print(f'Train: {len(trn_files)} files   Val: {len(val_files)} files')

# ── Re-instantiate fresh model + scheduler ───────────────────────────────────
model     = JointRelationshipModel(d_embed=64).to(device)
loss_fn   = NTXentLoss(temperature=TEMPERATURE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS, eta_min=LR / 20)


# ── Batch loader ──────────────────────────────────────────────────────────────
def load_batch(paths: list, device: torch.device):
    """Load a list of .pt files → (seqs [B,T,24,9], joint_idx [B*24])."""
    seqs = []
    for p in paths:
        d = torch.load(p, map_location='cpu', weights_only=False)
        seqs.append(d['vimu']['vimu_joints'])       # [300, 24, 9]
    seqs = torch.stack(seqs).to(device)             # [B, 300, 24, 9]
    B, T, N, C = seqs.shape
    seqs_flat  = seqs.permute(0, 2, 1, 3).reshape(B * N, T, C)   # [B*24, T, 9]
    joint_idx  = torch.arange(N, device=device).repeat(B)         # [B*24]
    return seqs_flat, joint_idx


# ── One epoch ─────────────────────────────────────────────────────────────────
def run_epoch(files: list, train: bool) -> float:
    model.train(train)
    ctx  = torch.enable_grad() if train else torch.no_grad()
    total, steps = 0.0, 0

    random.shuffle(files)
    with ctx:
        for start in range(0, len(files), BATCH_SIZE):
            batch_paths = files[start: start + BATCH_SIZE]
            seqs_flat, joint_idx = load_batch(batch_paths, device)

            if train:
                optimizer.zero_grad()

            embeddings = model.encode(seqs_flat)
            loss       = loss_fn(embeddings, joint_idx)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total += loss.item()
            steps += 1

    return total / max(steps, 1)


# ── Training loop ─────────────────────────────────────────────────────────────
trn_losses, val_losses = [], []
best_val   = float('inf')
t0         = time.time()

print(f'{"Epoch":>6}  {"Train loss":>12}  {"Val loss":>10}  {"LR":>10}  {"Time":>8}')
print('-' * 56)

for epoch in range(1, N_EPOCHS + 1):
    t_ep   = time.time()
    trn_l  = run_epoch(trn_files, train=True)
    val_l  = run_epoch(val_files, train=False)
    scheduler.step()

    trn_losses.append(trn_l)
    val_losses.append(val_l)

    lr_now = scheduler.get_last_lr()[0]
    elapsed = time.time() - t_ep
    print(f'{epoch:>6}  {trn_l:>12.5f}  {val_l:>10.5f}  {lr_now:>10.2e}  {elapsed:>7.1f}s')

    if val_l < best_val:
        best_val = val_l
        torch.save({
            'epoch':       epoch,
            'model_state': model.state_dict(),
            'optim_state': optimizer.state_dict(),
            'val_loss':    best_val,
            'd_embed':     64,
        }, CKPT_PATH)

print(f'\nTotal time : {(time.time() - t0)/60:.1f} min')
print(f'Best val loss : {best_val:.5f}  (saved → {CKPT_PATH})')

# ── Loss curve ────────────────────────────────────────────────────────────────
epochs_ax = range(1, N_EPOCHS + 1)
fig, ax   = plt.subplots(figsize=(9, 4))
ax.plot(epochs_ax, trn_losses, label='Train', color='steelblue',  linewidth=1.5)
ax.plot(epochs_ax, val_losses, label='Val',   color='darkorange', linewidth=1.5)
ax.axvline(val_losses.index(best_val) + 1, color='crimson', linestyle='--',
           linewidth=1, label=f'best val @ epoch {val_losses.index(best_val)+1}')
ax.set_xlabel('Epoch')
ax.set_ylabel('NT-Xent loss')
ax.set_title('Joint relationship model — training curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()